In [1]:
import pandas as pd
import os
import numpy as np
import re

In [2]:
pd.set_option('display.max_columns', None)

### <span style="color:#FF6347;">**READ**</span> file

In [ ]:
path = "P:\Dataset\MO-DBT-data-curation"

In [ ]:
file_path = os.path.join(path,'dicom_tag' + ".xlsx")
dicom = pd.read_excel(file_path)

In [ ]:
dicom[dicom["Series"]=="DBT"]

,PatientID,PatientBirthDate,PatientAge,AccessionNumber,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,Manufacturer,ManufacturerModelName,SliceThickness,Exposure,Rows,Columns,PixelSpacing,FolderPath
39,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088364\0.088364,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
40,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088500\0.088500,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
41,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088500\0.088500,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
42,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R MLO Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088500\0.088500,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
75,4330116791,1963-07-01,56.0,61499674,2019-12-17,DIAG,L,DBT,CC,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.106407\0.106407,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137766,4339601084,1972-07-01,46.0,77038224,2018-11-07,DIAG,L,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.106192\0.106192,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
137767,4339601084,1972-07-01,46.0,77038224,2018-11-07,DIAG,R,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.107029\0.107029,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
137768,4339601084,1972-07-01,46.0,77038224,2018-11-07,DIAG,R,DBT,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R MLO Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.106860\0.106860,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
137775,4339601084,1972-07-01,46.0,65758752,2019-05-16,DIAG,R,DBT,CC,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.106843\0.106843,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [ ]:
dicom[dicom["Series"]=="DBT"]["PatientID"].unique().size

2323

In [ ]:
file_path = os.path.join(path,'R3Data/dicomtocsv_series' + ".csv")
dicom_series = pd.read_csv(file_path)

file_path = os.path.join(path,'R3Data/dicomtocsv_study' + ".csv")
dicom_study = pd.read_csv(file_path)

In [ ]:
pids_dicom_plus_series = set(dicom_series['PatientID'].unique())
pids_dicom_plus_study = set(dicom_study['PatientID'].unique())
pids_dicom = set(dicom['PatientID'].unique())

In [ ]:
print("Number of patients we have image available: ", len(pids_dicom), 
"\nNumber of patients in R3 series and study: ", len(pids_dicom_plus_series), 
"\nDiscrpency between R3 series vs R3 study: ", len(pids_dicom_plus_series - pids_dicom_plus_study), 
"\nNumber of patients that we don't have corresponding R3 series: ", len(pids_dicom - pids_dicom_plus_series), 
"\nNumber of patients that we don't have corresponding R3 study: ", len(pids_dicom - pids_dicom_plus_study)) 

Number of patients we have image available:  5655 
Number of patients in R3 series and study:  5515 
Discrpency between R3 series vs R3 study:  0 
Number of patients that we don't have corresponding R3 series:  140 
Number of patients that we don't have corresponding R3 study:  140


In [ ]:
df = dicom.copy()

## **Non-imaging data** <span style="color:red">: CHECK DISCREPENCY </span>
### <span style="color:red">  </span>

### <span style="color:blue">**patient_demo**</span>

In [ ]:
demo_file_path_cancer = "P:/Dataset/MO-DBT-data-curation/Cancer/Cleaned/patient_demo.xlsx"
demo_file_path_control = "P:/Dataset/MO-DBT-data-curation/Control/Cleaned/patient_demo.xlsx"

demo_cancer = pd.read_excel(demo_file_path_cancer)[["PATIENT_STUDY_ID", "BIRTH_DATE"]]
demo_control = pd.read_excel(demo_file_path_control)[["PATIENT_STUDY_ID", "BIRTH_DATE"]]

In [26]:
demo =  pd.concat((demo_cancer, demo_control), axis=0)
demo = demo.drop_duplicates(subset=demo.columns.tolist(), keep = 'first').reset_index(drop = True)

In [27]:
demo.rename(columns={'PATIENT_STUDY_ID': 'PatientID', 'BIRTH_DATE': 'PatientBirthDate'}, inplace=True)

In [ ]:
df_birth = pd.merge(df, demo, on='PatientID', how='left')

In [29]:
df_birth['PatientID'].nunique(), df_birth.loc[df_birth['PatientBirthDate'].notna(), 'PatientID'].nunique(), df_birth.loc[df_birth['PatientBirthDate'].isna(), 'PatientID'].nunique()

(5655, 5539, 116)

### <span style="color:blue">**enteredit_findings**</span>

In [ ]:
birads_file_path_cancer = "P:/Dataset/MO-DBT-data-curation/Cancer/Cleaned/enteredit_findings.xlsx"
birads_file_path_control = "P:/Dataset/MO-DBT-data-curation/Control/Cleaned/enteredit_findings.xlsx"

birads_cancer = pd.read_excel(birads_file_path_cancer)[["PATIENT_STUDY_ID", "ACCESSION_NUMBER", "COMPOSITION_NAME", "FINDING_CATEGORY"]]
birads_control = pd.read_excel(birads_file_path_control)[["PATIENT_STUDY_ID", "ACCESSION_NUMBER", "COMPOSITION_NAME", "FINDING_CATEGORY"]]

In [31]:
birads =  pd.concat((birads_cancer, birads_control), axis=0)
# birads =  birads_cancer.copy()
birads = birads.drop_duplicates(subset=birads.columns.tolist(), keep = 'first').reset_index(drop = True)

In [32]:
birads.rename(columns={'PATIENT_STUDY_ID': 'PatientID', 'ACCESSION_NUMBER': 'AccessionNumber'}, inplace=True)

In [ ]:
df_birads = pd.merge(df, birads, on=['PatientID', 'AccessionNumber'], how='left')

In [34]:
df_birads.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'Manufacturer',
       'ManufacturerModelName', 'SliceThickness', 'Exposure', 'Rows',
       'Columns', 'PixelSpacing', 'COMPOSITION_NAME', 'FINDING_CATEGORY'],
      dtype='object')

In [35]:
df_birads['PatientID'].nunique(), df_birads.loc[df_birads['FINDING_CATEGORY'].notna(), 'PatientID'].nunique(), df_birads.loc[df_birads['FINDING_CATEGORY'].isna(), 'PatientID'].nunique()

(5655, 5536, 119)

### <span style="color:blue">**risk_factors**</span>

In [ ]:
risk_file_path_cancer = "P:/Dataset/MO-DBT-data-curation/Cancer/Cleaned/risk_factors.xlsx"
risk_file_path_control = "P:/Dataset/MO-DBT-data-curation/Control/Cleaned/risk_factors.xlsx"

risk_cancer = pd.read_excel(birads_file_path_cancer)[["PATIENT_STUDY_ID", "ACCESSION_NUMBER"]]
risk_control = pd.read_excel(birads_file_path_control)[["PATIENT_STUDY_ID", "ACCESSION_NUMBER"]]

In [37]:
risk =  pd.concat((risk_cancer, risk_control), axis=0)
# risk =  risk_cancer.copy()
risk = risk.drop_duplicates(subset=risk.columns.tolist(), keep = 'first').reset_index(drop = True)
risk['Risk'] = 1 

In [38]:
risk.rename(columns={'PATIENT_STUDY_ID': 'PatientID', 'ACCESSION_NUMBER': 'AccessionNumber'}, inplace=True)

In [ ]:
df_risk = pd.merge(df, risk, on=['PatientID', 'AccessionNumber'], how='left')

In [40]:
df_risk.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'Manufacturer',
       'ManufacturerModelName', 'SliceThickness', 'Exposure', 'Rows',
       'Columns', 'PixelSpacing', 'Risk'],
      dtype='object')

In [41]:
df_risk['PatientID'].nunique(), df_risk.loc[df_risk['Risk'].notna(), 'PatientID'].nunique(), df_risk.loc[df_risk['Risk'].isna(), 'PatientID'].nunique()

(5655, 5536, 119)

### <span style="color:blue">**vitals**</span>

In [ ]:
vital_file_path_cancer = "P:/Dataset/MO-DBT-data-curation/Cancer/Cleaned/vitals.xlsx"
vital_file_path_control = "P:/Dataset/MO-DBT-data-curation/Control/Cleaned/vitals.xlsx"

vital_cancer = pd.read_excel(vital_file_path_cancer)[["PATIENT_STUDY_ID"]]
vital_control = pd.read_excel(vital_file_path_control)[["PATIENT_STUDY_ID"]]

In [43]:
vital =  pd.concat((vital_cancer, vital_control), axis=0)
# vital =  vital_cancer.copy()
vital = vital.drop_duplicates(subset=vital.columns.tolist(), keep = 'first').reset_index(drop = True)
vital['vital'] = 1 

In [44]:
vital.rename(columns={'PATIENT_STUDY_ID': 'PatientID'}, inplace=True)

In [ ]:
df_vital = pd.merge(df, vital, on=['PatientID'], how='left')

In [46]:
df_vital.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'Manufacturer',
       'ManufacturerModelName', 'SliceThickness', 'Exposure', 'Rows',
       'Columns', 'PixelSpacing', 'vital'],
      dtype='object')

In [47]:
df_vital['PatientID'].nunique(), df_vital.loc[df_vital['vital'].notna(), 'PatientID'].nunique(), df_vital.loc[df_vital['vital'].isna(), 'PatientID'].nunique()

(5655, 5411, 244)

### <span style="color:blue">**procedure_notes**</span>

In [ ]:
procedure_notes_file_path_cancer = "P:/Dataset/MO-DBT-data-curation/Cancer/Cleaned/procedure_notes.xlsx"

procedure_notes_cancer = pd.read_excel(procedure_notes_file_path_cancer)[["PATIENT_STUDY_ID"]]

In [49]:
procedure_notes =  procedure_notes_cancer.copy()
procedure_notes = procedure_notes.drop_duplicates(subset=procedure_notes.columns.tolist(), keep = 'first').reset_index(drop = True)
procedure_notes['procedure_notes'] = 1 

In [50]:
procedure_notes.rename(columns={'PATIENT_STUDY_ID': 'PatientID'}, inplace=True)

In [ ]:
df_procedure_notes = pd.merge(df, procedure_notes, on=['PatientID'], how='left')

In [52]:
df_procedure_notes.columns

Index(['PatientID', 'StudyDate', 'StudyDescription', 'AccessionNumber',
       'SeriesNumber', 'SeriesDescription', 'FolderPath', 'Manufacturer',
       'ManufacturerModelName', 'SliceThickness', 'Exposure', 'Rows',
       'Columns', 'PixelSpacing', 'procedure_notes'],
      dtype='object')

In [53]:
df_procedure_notes['PatientID'].nunique(), df_procedure_notes.loc[df_procedure_notes['procedure_notes'].notna(), 'PatientID'].nunique(), df_procedure_notes.loc[df_procedure_notes['procedure_notes'].isna(), 'PatientID'].nunique()

(5655, 766, 4889)